# Exemplo Prático: Divisão, Treino, Validação e Limiares

Neste notebook, vamos usar as nossas funções que foram desenvolvidas na pasta `src/` para simular todo o fluxo técnico inicial do Hackathon:

1. **Simular** uma base de dados temporal aleatória (para que o código já possa ser rodado agora).
2. Fazer o **Split Temporal** (Treino / Validação / Teste) preservando vazamento cronológico.
3. **Treinar** um modelo simples só para gerar predições probabilísticas (`prob_fpd`).
4. Calcular as nossas **Métricas Customizadas** focadas no regulamento (ROC_AUC, PR_AUC, KS).
5. Criar e aplicar as **Faixas de Risco** (Política de Cobrança).

In [ ]:
import sys
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# 1. Adicionando o diretório pai e a pasta src/ no path para podermos importar as funções
sys.path.append('../src')

from validation import temporal_split
from metrics import calculate_metrics
from thresholds import define_risk_bands

## 1. Simulação dos Dados de FPD
Criando uma base de dados _dummy_ com datas sequenciais. Para fins de simulação, 
cerca de 15% das operações serão classificadas como FPD = 1.

In [ ]:
np.random.seed(42)
n_samples = 10000

# Datas distribuídas sequencialmente (simulando 6 meses de operações via TMB)
dates = pd.date_range(start='2025-01-01', periods=n_samples, freq='h')

# Criando 2 Features aleatórias (que podem imitar um bureau score, etc)
X1 = np.random.randn(n_samples)
X2 = np.random.randn(n_samples) * 0.5 + 2

# Variável Target (Y): Gerando um certo padrão de risco escondido
# A chance de Inadimplência (FPD) aumenta se X1 é maior e X2 é menor
risco_oculto = (X1 - X2) + np.random.randn(n_samples)
prob_real = 1 / (1 + np.exp(-risco_oculto)) # Sigmoid converte valor bruto em (0.0 até 1.0)
y = (prob_real > 0.8).astype(int) # Os top piores ~15% viram Classe 1 (FPD)

df = pd.DataFrame({
    'data_compra': dates,
    'score_bureau': X1,
    'valor_compra': X2,
    'fpd_target': y
})

print(f"Total de linhas simuladas: {len(df)}")
print(f"Taxa de FPD na base (%): {df['fpd_target'].mean()*100:.2f}%")
df.head()

## 2. Divisão Temporal (Temporal Split)
Evita vazamento! Treina no Passado, Valida no Meio, e simula Produção no Futuro.

In [ ]:
# Split obrigatório de 70/15/15 mantendo a linha do tempo estrita
df_train, df_val, df_test = temporal_split(df, date_column='data_compra', test_size=0.15, val_size=0.15)

print(f"TREINO (Passado) -> de {df_train['data_compra'].min()} a {df_train['data_compra'].max()} ({len(df_train)} obs)")
print(f"VALIDA (Presente) -> de {df_val['data_compra'].min()} a {df_val['data_compra'].max()} ({len(df_val)} obs)")
print(f"TESTE  (Futuro) -> de {df_test['data_compra'].min()} a {df_test['data_compra'].max()} ({len(df_test)} obs)")

## 3. Treinamento de um Modelo Simples
Treinando na base `df_train` e pontuando (predict_proba) a base `df_val`.

In [ ]:
features = ['score_bureau', 'valor_compra']

# Inicializamos e Treinamos o classificador com o Dataset de TREINO (70%)
model = LogisticRegression()
model.fit(df_train[features], df_train['fpd_target'])

# Predições na base de VALIDAÇÃO (Pegamos a probabilidade da coluna 1, que indica FPD)
y_val_prob = model.predict_proba(df_val[features])[:, 1]
y_val_true = df_val['fpd_target']

print("As 5 primeiras probabilidades da nossa lista de clientes:")
y_val_prob[:5]

## 4. Avaliando as Métricas do Regulamento
Verificando ROC, KS e PR Area under Curve na base de Validação.

In [ ]:
# Setamos um corte (threshold) baixo só para ver quanto Recall ele pega se acionarmos 30% da base
resultados = calculate_metrics(y_val_true, y_val_prob, threshold=0.3)

print("Desempenho Geral do Modelo (Validação):\n")
for metric, value in resultados.items():
    print(f"- {metric}: {value:.4f}")

## 5. Aplicação da Política de Risco (Faixas de Cobrança)
Mapeando a pontuação decimal de `0 a 1.0` para rótulos fáceis (`Baixo, Médio, Alto, Crítico`) 
para montar as réguas de cobrança automatizadas via WhatsApp, SMS ou Agente Virtual.

In [ ]:
# Passa o Array NumPy de probabilidades para a função, que retorna um Dataframe rotulado
df_faixas = define_risk_bands(y_val_prob)

# Juntando as bandas criadas ao dataframe original de validação
df_val_final = df_val.copy()
df_val_final['prob_fpd'] = y_val_prob
df_val_final['faixa_risco'] = df_faixas['faixa_risco'].values 

# Vendo o total de clientes impactados e a taxa real de FPD por faixa
resumo = df_val_final.groupby('faixa_risco').agg(
    quantidade_clientes=('fpd_target', 'count'),
    tx_inadimplencia_real_fpd=('fpd_target', 'mean'),
    media_probabilidade_modelo=('prob_fpd', 'mean')
).reset_index()

# Ordenar do Risco Baixo para o Crítico
sorter = ['Baixo', 'Médio', 'Alto', 'Crítico']
resumo['faixa_risco'] = pd.Categorical(resumo['faixa_risco'], categories=sorter, ordered=True)
resumo = resumo.sort_values('faixa_risco')

resumo.style.format({
    'tx_inadimplencia_real_fpd': '{:.2%}',
    'media_probabilidade_modelo': '{:.2%}'
}).background_gradient(cmap='Reds', subset=['tx_inadimplencia_real_fpd'])
